# 05 — RAG from Scratch (Phase 5, Milestone 5)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udaysharmadev/Ai-Roadmap/blob/main/notebooks/05_rag_minimal.ipynb)

**Maps to:** `docs/rag-roadmap.md` + `README Milestone 5`  
**Data:** `data/samples/rag_docs/` (4 short docs, offline)  
**Keys required:** none — hashed embeddings + extractive answers run anywhere.

Flow: chunk → embed → index → retrieve → generate → evaluate. This notebook *is* the worked solution for the PDF Q&A chatbot (Streamlit template serves the same index).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_roadmap.embeddings import HashingEmbedder
from ai_roadmap.rag_pipeline import (
    answer_query,
    build_index,
    hit_at_k,
    load_docs_from_dir,
    retrieve,
)
from ai_roadmap.text_chunking import chunk_text

DOCS = ROOT / "data/samples/rag_docs"
print(f"docs dir: {DOCS.exists()}")

## 1. Chunking — why overlap matters

Without overlap a key sentence can split across chunks and never match. Overlap keeps context.

In [ ]:
demo = "a b c d e f g h"
print([c.text for c in chunk_text(demo, chunk_size=4, overlap=0)])
print([c.text for c in chunk_text(demo, chunk_size=4, overlap=2)])

docs = load_docs_from_dir(DOCS)
print(f"loaded {len(docs)} docs: {[d['id'] for d in docs]}")
for d in docs:
    n = len(chunk_text(d["text"], chunk_size=120, overlap=20))
    print(f"  {d['id']}: {len(d['text'].split())} words -> {n} chunk(s)")

## 2. Embeddings — same words, high cosine

Hashed bag-of-words: paraphrases sharing vocabulary score high (like real dense retrievers on this scale).

In [ ]:
import numpy as np

emb = HashingEmbedder(dim=256)
q = emb.embed_query("hybrid search vector database")
candidates = emb.embed_texts([
    "hybrid search combines sparse BM25 with dense vectors then reranks",
    "convolutional networks use relu pooling and backpropagation",
])
sims = candidates @ q
print(f"rag-doc sim: {sims[0]:.3f} | cnn-doc sim: {sims[1]:.3f}")
assert sims[0] > sims[1], "retrieval should prefer the RAG passage"
print("ranking check passed ✅")

## 3. Build the index (one call)

In [ ]:
store, embedder, chunks = build_index(docs, chunk_size=120, overlap=20)
print(f"chunks: {len(chunks)} | dim: {store.dim} | ids[0]: {store.ids[0]}")

## 4. Retrieve — one query per topic

In [ ]:
queries = {
    "How does hybrid search with reranking work?": "rag_systems",
    "What is the ReAct pattern with tools?": "ai_agents",
    "Why does ReLU help backpropagation?": "deep_learning",
    "How to prevent overfitting with cross validation?": "ml_basics",
}
for q, expected in queries.items():
    hits = retrieve(store, embedder, q, k=2)
    top = hits[0]["id"]
    mark = "✅" if top.startswith(expected) else "❌"
    print(f"{mark} Q: {q[:50]}… -> {top} ({hits[0]['score']:.3f})")

## 5. Generate — extractive answers with citations (no LLM key)

In [ ]:
result = answer_query(store, embedder, "How does hybrid search with reranking work?", k=2)
print(result["answer"])
print()
print(f"citations: {result['citations']}")

## 6. Evaluate — hit@k over the topic queries

In [ ]:
scores = []
for q, expected in queries.items():
    hits = retrieve(store, embedder, q, k=2)
    scores.append(hit_at_k([h["id"] for h in hits], [expected], k=2))
print(f"hit@2: {sum(scores) / len(scores):.2f} over {len(scores)} queries")
assert sum(scores) / len(scores) >= 0.75, "retrieval quality regressed"
print("eval check passed ✅")

## 7. Save the index (Streamlit template serves this)

In [ ]:
out = ROOT / "outputs"
out.mkdir(exist_ok=True)
base = store.save(out / "rag_demo_index")
print(f"saved: {base.with_suffix('.npz').name} + {base.with_suffix('.json').name}")

# Reload sanity check
from ai_roadmap.vector_store import SimpleVectorStore

reloaded = SimpleVectorStore.load(out / "rag_demo_index")
assert len(reloaded) == len(store)
print(f"reload check passed ✅ ({len(reloaded)} vectors)")

## ✅ Milestone-5 checks

- [x] Docs chunked with overlap + provenance ids
- [x] Offline embeddings rank the right topic first
- [x] Top-k retrieval + cited extractive answers
- [x] hit@k eval ≥ 0.75 + saved index artefact

**Next (Chunk 5):** `projects/05_pdf_rag_chatbot/` — the same pipeline behind a file-upload UI.  
**Try live:** `templates/streamlit_rag/` serves this index as a chat app.